# 气候可视化 - CO₂排放分析

包含3个精选CO₂图表

---

## 环境设置

In [1]:
# 导入库
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 中文字体
plt.rcParams['font.sans-serif'] = ['STHeiti', 'PingFang SC', 'Hiragino Sans GB']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['savefig.dpi'] = 300

sns.set_style('whitegrid')

OUT_DIR = Path('output/figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('✓ 环境完成')

✓ 环境完成


## 数据加载

In [2]:
import sys
sys.path.append('code')
from data_loader import ClimateDataLoader

loader = ClimateDataLoader(data_dir='data/raw')
df_global = loader.load_global_temperatures()
df_co2 = loader.load_co2_data()

if df_co2 is None:
    print('⚠️ CO₂数据未找到')
else:
    print('✅ 数据加载完成')

正在加载全球温度数据: data/raw/GlobalTemperatures.csv
✓ 加载完成：3192 条记录
正在加载CO2数据: data/raw/co2/co2_emissions_kt_by_country.csv
  数据列: ['country_code', 'country_name', 'year', 'value']
✓ 加载完成：13953 条记录
  时间范围：1960 - 2019
  CO2列：value
  国家数量：256
✅ 数据加载完成


---

## 图表 1: CO₂时间序列

主要排放国CO₂趋势

In [3]:
# 图表1: CO₂时间序列
if df_co2 is not None:
    # 选择TOP10排放国
    latest_year = df_co2['year'].max()
    df_latest = df_co2[df_co2['year'] == latest_year]
    top10_countries = df_latest.nlargest(10, 'co2_emission')['country_name'].tolist()
    
    df_top10 = df_co2[df_co2['country_name'].isin(top10_countries)]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = sns.color_palette('viridis', 10)
    for i, country in enumerate(top10_countries):
        country_data = df_top10[df_top10['country_name'] == country]
        ax.plot(country_data['year'], country_data['co2_emission'] / 1000,
               linewidth=2.5, label=country, color=colors[i], alpha=0.8)
    
    ax.set_xlabel('年份', fontsize=13, weight='bold')
    ax.set_ylabel('CO₂排放 (千吨)', fontsize=13, weight='bold')
    ax.set_title('主要排放国CO₂时间序列', fontsize=15, weight='bold', pad=15)
    ax.legend(loc='best', fontsize=10, ncol=2, frameon=True, shadow=True)
    ax.grid(True, alpha=0.3)
    sns.despine()
    
    plt.tight_layout()
    plt.savefig(OUT_DIR / '06_co2_time_series_lines.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('✓ 图表1完成')
else:
    print('⚠️ 跳过图表1')

KeyError: 'country_name'

---

## 图表 2: CO₂与温度关系

Hex密度图展示相关性

In [ ]:
# 图表2: CO₂ vs 温度
if df_co2 is not None:
    # 数据准备
    df_co2_yearly = df_co2.groupby('year')['co2_emission'].sum().reset_index()
    df_temp_yearly = df_global.groupby('year')['LandAverageTemperature'].mean().reset_index()
    df_merged = df_co2_yearly.merge(df_temp_yearly, on='year').dropna()
    
    # JointPlot
    g = sns.jointplot(
        x=df_merged['co2_emission'] / 1e6,
        y=df_merged['LandAverageTemperature'],
        kind='hex',
        color='#4CB391',
        height=9,
        ratio=5,
        marginal_kws=dict(bins=30, fill=True)
    )
    
    # 回归线
    z = np.polyfit(df_merged['co2_emission'] / 1e6, df_merged['LandAverageTemperature'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_merged['co2_emission'].min() / 1e6,
                        df_merged['co2_emission'].max() / 1e6, 100)
    r_sq = np.corrcoef(df_merged['co2_emission'], df_merged['LandAverageTemperature'])[0,1]**2
    g.ax_joint.plot(x_line, p(x_line), 'r-', linewidth=2.5,
                   alpha=0.8, label=f'R² = {r_sq:.3f}')
    
    g.set_axis_labels('全球CO₂排放 (百万吨)', '全球温度 (°C)',
                     fontsize=12, weight='bold')
    g.fig.suptitle('CO₂排放与温度的密集关系', fontsize=14, weight='bold', y=1.02)
    g.ax_joint.legend(fontsize=10)
    
    plt.savefig(OUT_DIR / '04_co2_temp_hexjoint.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('✓ 图表2完成')
else:
    print('⚠️ 跳过图表2')

---

## 图表 3: CO₂排放分布

人均vs总量JointPlot

In [ ]:
# 图表3: CO₂分布
if df_co2 is not None:
    # 使用最新年份数据
    latest_year = df_co2['year'].max()
    df_latest = df_co2[df_co2['year'] == latest_year].copy()
    
    # 计算人均排放（假设有人口数据，否则用简化版本）
    df_plot = df_latest[(df_latest['co2_emission'] > 0)].copy()
    
    # 如果有co2_per_capita列
    if 'co2_per_capita' in df_plot.columns:
        df_plot = df_plot.dropna(subset=['co2_per_capita', 'co2_emission'])
        df_plot = df_plot[(df_plot['co2_per_capita'] > 0)]
        
        g = sns.jointplot(
            data=df_plot,
            x='co2_per_capita',
            y='co2_emission',
            kind='scatter',
            height=9,
            ratio=4,
            color='#6A5ACD',
            alpha=0.6,
            marginal_kws=dict(bins=25, fill=True)
        )
        
        g.set_axis_labels('人均CO₂排放 (吨/人)', '国家总排放 (吨)',
                         fontsize=12, weight='bold')
    else:
        # 简化版本：仅展示总排放分布
        fig, ax = plt.subplots(figsize=(12, 8))
        top_countries = df_plot.nlargest(30, 'co2_emission')
        colors = sns.color_palette('viridis', len(top_countries))
        ax.barh(range(len(top_countries)), top_countries['co2_emission'] / 1000,
               color=colors, edgecolor='white')
        ax.set_yticks(range(len(top_countries)))
        ax.set_yticklabels(top_countries['country_name'])
        ax.set_xlabel('CO₂排放 (千吨)', fontsize=13, weight='bold')
        ax.set_title(f'TOP30排放国 ({latest_year}年)', fontsize=15, weight='bold')
        ax.grid(axis='x', alpha=0.3)
        sns.despine()
    
    plt.tight_layout()
    plt.savefig(OUT_DIR / '05_co2_distribution_combo.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('✓ 图表3完成')
else:
    print('⚠️ 跳过图表3')